### EXPERIMENT — CHURN HORIZON 90 NGÀY

> Notebook giữ nguyên luồng **purchase-event snapshot**. Một dòng tương ứng với một lần mua thực tế của một `customer_unique_id`.

- **Churn chính:** `churn_90`.
- `churn_90 = 0`: khách có lần mua tiếp theo trong vòng 90 ngày sau `snapshot_at`.
- `churn_90 = 1`: không có lần mua tiếp theo trong 90 ngày và có đủ 90 ngày quan sát tương lai.
- Không đủ 90 ngày quan sát: `CENSORED`, không đưa vào model.
- Các ngưỡng 120/180/365 vẫn được tạo để đối chiếu, nhưng không dùng làm target chính.


In [1]:
%pip -q install pyspark hvplot panel jupyter_bokeh

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("OlistChurnPurchaseEvent").getOrCreate()

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Data directory     :", data_dir)
print("Processed directory:", processed_dir)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 14.9 MB/s eta 0:00:00
Mounted at /content/drive
Data directory     : /content/drive/MyDrive/BigData/DA-BigData/code/data
Processed directory: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed


In [2]:
from pathlib import Path
from pyspark.sql import functions as F

# ============================================================
# 1. KIỂM TRA FILE ĐẦU VÀO
# ============================================================

orders_prepared_path = processed_dir / "olist_orders_prepared.csv"
orders_raw_path = data_dir / "olist_orders_dataset.csv"
customers_path = data_dir / "olist_customers_dataset.csv"
items_path = data_dir / "olist_order_items_dataset.csv"
payments_path = data_dir / "olist_order_payments_dataset.csv"
reviews_path = data_dir / "olist_order_reviews_dataset.csv"
products_prepared_path = processed_dir / "olist_products_prepared.csv"
payment_audit_path = processed_dir / "olist_order_payments_prepared.csv"

required_base = [
    customers_path,
    items_path,
    payments_path,
    reviews_path,
]

if not orders_prepared_path.exists() and not orders_raw_path.exists():
    raise FileNotFoundError(
        "Không tìm thấy cả olist_orders_prepared.csv và olist_orders_dataset.csv"
    )

missing_base = [str(p) for p in required_base if not p.exists()]
if missing_base:
    raise FileNotFoundError(f"Thiếu file đầu vào: {missing_base}")

orders_path = orders_prepared_path if orders_prepared_path.exists() else orders_raw_path
print("Orders source:", orders_path)

# ============================================================
# 2. ĐỌC ORDERS + CUSTOMERS
# ============================================================

orders = (
    spark.read.option("header", True)
    .csv(str(orders_path))
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

customers = (
    spark.read.option("header", True)
    .csv(str(customers_path))
    .select("customer_id", "customer_unique_id")
)

if customers.count() != customers.select("customer_id").distinct().count():
    raise ValueError("customer_id bị trùng trong bảng customers")

if orders.count() != orders.select("order_id").distinct().count():
    raise ValueError("order_id bị trùng trong bảng orders")

# ============================================================
# 3. CHỈ GIỮ ĐƠN DELIVERED VÀ GẮN CUSTOMER_UNIQUE_ID
# ============================================================

purchases = (
    orders
    .filter(F.col("order_status") == "delivered")
    .join(customers, "customer_id", "inner")
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("delivered_at", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_delivery_at", F.to_timestamp("order_estimated_delivery_date"))
    .filter(
        F.col("customer_unique_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("purchase_at").isNotNull()
    )
    .select(
        "customer_unique_id",
        "order_id",
        "purchase_at",
        "delivered_at",
        "estimated_delivery_at"
    )
    .dropDuplicates(["order_id"])
    .cache()
)

purchase_rows = purchases.count()
unique_customers = purchases.select("customer_unique_id").distinct().count()
data_end_at = purchases.agg(F.max("purchase_at").alias("data_end_at")).first()["data_end_at"]

if data_end_at is None:
    raise ValueError("Không tìm thấy purchase_at hợp lệ")

print("Số đơn delivered hợp lệ:", purchase_rows)
print("Số customer_unique_id   :", unique_customers)
print("Ngày mua cuối dữ liệu   :", data_end_at)


Orders source: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_orders_prepared.csv
Số đơn delivered hợp lệ: 96478
Số customer_unique_id   : 93358
Ngày mua cuối dữ liệu   : 2018-08-29 15:00:37


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. SẮP XẾP CHUỖI MUA THEO CUSTOMER_UNIQUE_ID
# ============================================================

w_order = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(F.col("purchase_at"), F.col("order_id"))
)

w_history = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(F.col("purchase_at"), F.col("order_id"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

purchase_events = (
    purchases
    .withColumn("purchase_sequence", F.row_number().over(w_order))
    .withColumn("previous_purchase_at", F.lag("purchase_at").over(w_order))
    .withColumn("next_purchase_at", F.lead("purchase_at").over(w_order))
    .withColumn("first_purchase_at", F.first("purchase_at").over(w_history))
    .withColumn(
        "days_since_previous_purchase",
        F.when(
            F.col("previous_purchase_at").isNotNull(),
            (
                F.unix_timestamp("purchase_at")
                - F.unix_timestamp("previous_purchase_at")
            ) / 86400.0
        )
    )
    .withColumn(
        "gap_days",
        F.when(
            F.col("next_purchase_at").isNotNull(),
            (
                F.unix_timestamp("next_purchase_at")
                - F.unix_timestamp("purchase_at")
            ) / 86400.0
        )
    )
    .withColumn(
        "customer_tenure_days",
        (
            F.unix_timestamp("purchase_at")
            - F.unix_timestamp("first_purchase_at")
        ) / 86400.0
    )
    .withColumn(
        "avg_interpurchase_days_to_snapshot",
        F.avg("days_since_previous_purchase").over(w_history)
    )
    .withColumn(
        "max_interpurchase_days_to_snapshot",
        F.max("days_since_previous_purchase").over(w_history)
    )
    .withColumnRenamed("order_id", "snapshot_order_id")
    .withColumnRenamed("purchase_at", "snapshot_at")
    .withColumn("snapshot_date", F.to_date("snapshot_at"))
)

# ============================================================
# 2. FOLLOW-UP VÀ NHÃN CHURN
# ============================================================

DATA_END = F.lit(data_end_at).cast("timestamp")

for threshold in [90, 120, 180, 365]:
    followup_col = f"followup_end_{threshold}"
    churn_col = f"churn_{threshold}"
    status_col = f"label_status_{threshold}"
    full_col = f"has_full_followup_{threshold}"

    purchase_events = (
        purchase_events
        .withColumn(
            followup_col,
            F.expr(f"snapshot_at + INTERVAL {threshold} DAYS")
        )
        .withColumn(
            full_col,
            F.when(F.col(followup_col) <= DATA_END, 1).otherwise(0)
        )
        .withColumn(
            churn_col,
            F.when(
                F.col("next_purchase_at").isNotNull()
                & (F.col("next_purchase_at") <= F.col(followup_col)),
                F.lit(0)
            )
            .when(
                F.col("next_purchase_at").isNotNull()
                & (F.col("next_purchase_at") > F.col(followup_col)),
                F.lit(1)
            )
            .when(
                F.col("next_purchase_at").isNull()
                & (F.col(full_col) == 1),
                F.lit(1)
            )
            .otherwise(F.lit(None).cast("int"))
        )
        .withColumn(
            status_col,
            F.when(F.col(churn_col) == 0, F.lit("NON_CHURN"))
            .when(
                (F.col(churn_col) == 1)
                & F.col("next_purchase_at").isNotNull(),
                F.lit(f"CHURN_GAP_GT_{threshold}")
            )
            .when(
                (F.col(churn_col) == 1)
                & F.col("next_purchase_at").isNull(),
                F.lit("CHURN_NO_RETURN")
            )
            .otherwise(F.lit("CENSORED"))
        )
    )

purchase_events = (
    purchase_events
    .withColumn(
        "followup_days_available",
        (
            F.unix_timestamp(DATA_END)
            - F.unix_timestamp("snapshot_at")
        ) / 86400.0
    )
    .cache()
)

# ============================================================
# 3. KIỂM TRA LOGIC NHÃN
# ============================================================

duplicate_event_keys = (
    purchase_events
    .groupBy("customer_unique_id", "snapshot_order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_sequence = (
    purchase_events
    .filter(
        F.col("next_purchase_at").isNotNull()
        & (F.col("next_purchase_at") < F.col("snapshot_at"))
    )
    .count()
)

invalid_90 = (
    purchase_events
    .filter(
        (
            (F.col("churn_90") == 0)
            & (
                F.col("next_purchase_at").isNull()
                | (F.col("next_purchase_at") > F.col("followup_end_90"))
            )
        )
        | (
            (F.col("churn_90") == 1)
            & F.col("next_purchase_at").isNull()
            & (F.col("has_full_followup_90") != 1)
        )
    )
    .count()
)

invalid_label_order = (
    purchase_events
    .filter(
        F.col("churn_90").isNotNull()
        & F.col("churn_120").isNotNull()
        & F.col("churn_180").isNotNull()
        & F.col("churn_365").isNotNull()
        & (
            (F.col("churn_90") < F.col("churn_120"))
            | (F.col("churn_120") < F.col("churn_180"))
            | (F.col("churn_180") < F.col("churn_365"))
        )
    )
    .count()
)

print("Duplicate customer + snapshot_order_id:", duplicate_event_keys)
print("next_purchase_at < snapshot_at         :", invalid_sequence)
print("Nhãn churn_90 không hợp lệ             :", invalid_90)
print("Sai thứ tự churn_90/120/180/365         :", invalid_label_order)

if any(x > 0 for x in [duplicate_event_keys, invalid_sequence, invalid_90, invalid_label_order]):
    raise ValueError("Kiểm tra logic snapshot/nhãn chưa đạt")

# ============================================================
# 4. LƯU TOÀN BỘ PURCHASE-EVENT SNAPSHOT, KỂ CẢ CENSORED
# ============================================================

snapshot_all_dir = processed_dir / "olist_customer_purchase_event_snapshots"

(
    purchase_events
    .repartition(8, "snapshot_date")
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(snapshot_all_dir))
)

print("Đã lưu toàn bộ event snapshot:", snapshot_all_dir)


Duplicate customer + snapshot_order_id: 0
next_purchase_at < snapshot_at         : 0
Nhãn churn_90 không hợp lệ             : 0
Sai thứ tự churn_90/120/180/365         : 0
Đã lưu toàn bộ event snapshot: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_customer_purchase_event_snapshots


### Cell 2 — Kiểm tra snapshot và giữ tập có nhãn 90 ngày

- Kiểm tra số purchase-event, số khách, churn/non-churn và số dòng bị censor.
- Target chính là `churn_90`.
- Chỉ giữ observation có `churn_90` xác định.
- Các dòng `CENSORED` không đi vào Feature Engineering / Model Training.


In [6]:
from pyspark.sql import functions as F

snapshot_all_dir = processed_dir / "olist_customer_purchase_event_snapshots"
snapshot_labeled_dir = processed_dir / "olist_customer_purchase_event_snapshots_labeled_90"

snapshots = (
    spark.read.option("header", True)
    .csv(str(snapshot_all_dir))
    .withColumn("snapshot_at", F.to_timestamp("snapshot_at"))
    .withColumn("snapshot_date", F.to_date("snapshot_date"))
    .withColumn("previous_purchase_at", F.to_timestamp("previous_purchase_at"))
    .withColumn("next_purchase_at", F.to_timestamp("next_purchase_at"))
    .withColumn("followup_end_90", F.to_timestamp("followup_end_90"))
    .withColumn("followup_end_120", F.to_timestamp("followup_end_120"))
    .withColumn("followup_end_180", F.to_timestamp("followup_end_180"))
    .withColumn("followup_end_90", F.to_timestamp("followup_end_90"))
    .withColumn("purchase_sequence", F.col("purchase_sequence").cast("int"))
    .withColumn("days_since_previous_purchase", F.col("days_since_previous_purchase").cast("double"))
    .withColumn("gap_days", F.col("gap_days").cast("double"))
    .withColumn("customer_tenure_days", F.col("customer_tenure_days").cast("double"))
    .withColumn("avg_interpurchase_days_to_snapshot", F.col("avg_interpurchase_days_to_snapshot").cast("double"))
    .withColumn("max_interpurchase_days_to_snapshot", F.col("max_interpurchase_days_to_snapshot").cast("double"))
    .withColumn("followup_days_available", F.col("followup_days_available").cast("double"))
    .withColumn("churn_90", F.col("churn_90").cast("int"))
    .withColumn("churn_120", F.col("churn_120").cast("int"))
    .withColumn("churn_90", F.col("churn_90").cast("int"))
    .cache()
)

# ============================================================
# 1. TỔNG QUAN EVENT SNAPSHOT
# ============================================================

summary = (
    snapshots
    .agg(
        F.count("*").alias("so_event_snapshot"),
        F.countDistinct("customer_unique_id").alias("so_khach"),
        F.sum(F.when(F.col("churn_90") == 0, 1).otherwise(0)).alias("non_churn_90"),
        F.sum(F.when(F.col("churn_90") == 1, 1).otherwise(0)).alias("churn_90"),
        F.sum(F.when(F.col("churn_90").isNull(), 1).otherwise(0)).alias("censored_90")
    )
)

print("TỔNG QUAN PURCHASE-EVENT SNAPSHOT")
summary.show(truncate=False)

print("TRẠNG THÁI NHÃN 90 NGÀY")
(
    snapshots
    .groupBy("label_status_90")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct("customer_unique_id").alias("so_khach")
    )
    .orderBy("label_status_90")
    .show(truncate=False)
)

# ============================================================
# 2. SỐ EVENT TRÊN MỖI KHÁCH
# ============================================================

events_per_customer = (
    snapshots
    .groupBy("customer_unique_id")
    .agg(F.count("*").alias("so_lan_mua"))
)

print("PHÂN BỐ SỐ LẦN MUA TRÊN MỖI KHÁCH")
(
    events_per_customer
    .groupBy("so_lan_mua")
    .agg(F.count("*").alias("so_khach"))
    .orderBy("so_lan_mua")
    .show(100, truncate=False)
)

# ============================================================
# 3. ĐỐI CHIẾU CUSTOMER-LEVEL: CHỈ LẦN MUA ĐẦU
# ============================================================

first_purchase_validation = (
    snapshots
    .filter((F.col("purchase_sequence") == 1) & F.col("churn_90").isNotNull())
    .agg(
        F.count("*").alias("khach_du_90_ngay"),
        F.sum(F.when(F.col("churn_90") == 0, 1).otherwise(0)).alias("mua_lai_trong_90"),
        F.sum(F.when(F.col("churn_90") == 1, 1).otherwise(0)).alias("khong_mua_lai_90")
    )
    .withColumn(
        "ty_le_churn_90_pct",
        F.round(100 * F.col("khong_mua_lai_90") / F.col("khach_du_90_ngay"), 2)
    )
)

print("ĐỐI CHIẾU LẦN MUA ĐẦU — CUSTOMER LEVEL")
first_purchase_validation.show(truncate=False)

# ============================================================
# 4. KHÁCH CÓ CẢ NON-CHURN VÀ CHURN Ở CÁC LẦN MUA KHÁC NHAU
# ============================================================

customer_label_mix = (
    snapshots
    .filter(F.col("churn_90").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.min("churn_90").alias("min_label"),
        F.max("churn_90").alias("max_label"),
        F.count("*").alias("so_event_co_nhan")
    )
)

both_label_customers = (
    customer_label_mix
    .filter((F.col("min_label") == 0) & (F.col("max_label") == 1))
    .count()
)

print("Số khách từng có cả non-churn và churn:", both_label_customers)

# ============================================================
# 5. GIỮ TẬP CÓ NHÃN 90 NGÀY
# ============================================================

labeled_90 = (
    snapshots
    .filter(F.col("churn_90").isNotNull())
    .cache()
)

label_distribution = (
    labeled_90
    .groupBy("churn_90")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct("customer_unique_id").alias("so_khach")
    )
    .withColumn(
        "ty_le_event_pct",
        F.round(100 * F.col("so_event") / F.sum("so_event").over(Window.partitionBy()), 2)
    )
    .orderBy("churn_90")
)

print("PHÂN BỐ NHÃN 90 NGÀY Ở CẤP EVENT")
label_distribution.show(truncate=False)

# ============================================================
# TEST-ONLY: KHÔNG LƯU KẾT QUẢ RA DISK
# Theo yêu cầu, block lưu tập labeled_90 được comment lại.
# labeled_90 vẫn tồn tại trong memory để kiểm tra / thử nghiệm.
# ============================================================

# (
#     labeled_90
#     .repartition(8, "snapshot_date")
#     .write
#     .mode("overwrite")
#     .option("header", True)
#     .csv(str(snapshot_labeled_dir))
# )
#
# print("Đã lưu tập có nhãn 90 ngày:", snapshot_labeled_dir)

# labeled_90.unpersist()
snapshots.unpersist()


TỔNG QUAN PURCHASE-EVENT SNAPSHOT
+-----------------+--------+------------+--------+-----------+
|so_event_snapshot|so_khach|non_churn_90|churn_90|censored_90|
+-----------------+--------+------------+--------+-----------+
|96478            |93358   |2148        |75871   |18459      |
+-----------------+--------+------------+--------+-----------+

TRẠNG THÁI NHÃN 90 NGÀY
+---------------+--------+--------+
|label_status_90|so_event|so_khach|
+---------------+--------+--------+
|CENSORED       |18459   |18459   |
|CHURN_GAP_GT_90|972     |944     |
|CHURN_NO_RETURN|74899   |74899   |
|NON_CHURN      |2148    |1953    |
+---------------+--------+--------+

PHÂN BỐ SỐ LẦN MUA TRÊN MỖI KHÁCH
+----------+--------+
|so_lan_mua|so_khach|
+----------+--------+
|1         |90557   |
|2         |2573    |
|3         |181     |
|4         |28      |
|5         |9       |
|6         |5       |
|7         |3       |
|9         |1       |
|15        |1       |
+----------+--------+

ĐỐI CHIẾU LẦN MU

DataFrame[customer_unique_id: string, snapshot_order_id: string, snapshot_at: timestamp, delivered_at: string, estimated_delivery_at: string, purchase_sequence: int, previous_purchase_at: timestamp, next_purchase_at: timestamp, first_purchase_at: string, days_since_previous_purchase: double, gap_days: double, customer_tenure_days: double, avg_interpurchase_days_to_snapshot: double, max_interpurchase_days_to_snapshot: double, snapshot_date: date, followup_end_90: timestamp, has_full_followup_90: string, churn_90: int, label_status_90: string, followup_end_120: timestamp, has_full_followup_120: string, churn_120: int, label_status_120: string, followup_end_180: timestamp, has_full_followup_180: string, churn_180: string, label_status_180: string, followup_end_365: string, has_full_followup_365: string, churn_365: string, label_status_365: string, followup_days_available: double]

### Cell 3 — Tạo feature tại snapshot và chia Train / Embargo / Test

- Feature chỉ sử dụng thông tin đã biết tại `snapshot_at`.
- Target chính: `churn_90`.
- Train kết thúc tại `2017-11-01`, test bắt đầu từ `2018-02-01`, tạo khoảng cách xấp xỉ 90 ngày.
- Kiểm tra `followup_end_90` của train phải nhỏ hơn thời điểm bắt đầu test để hạn chế leakage.
- Các feature lịch sử 30/90/180 ngày vẫn được giữ vì đây là cửa sổ feature, không phải churn horizon.


In [7]:
from pathlib import Path
from pyspark.sql import functions as F

feature_output_dir = processed_dir / "olist_customer_churn_features_temporal"

# Temporal split cho churn horizon 90 ngày
TRAIN_END = "2017-11-01"
TEST_START = "2018-02-01"

def read_csv(path, multiline=False):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", multiline)
        .option("escape", '"')
        .csv(str(path))
    )

# ============================================================
# 1. LẤY SNAPSHOT CÓ NHÃN 90 NGÀY
# Không đọc olist_customer_purchase_event_snapshots_labeled_90
# vì Cell 2 không lưu dataset này ra disk.
# ============================================================

if "labeled_90" not in globals():
    raise RuntimeError(
        "Không tìm thấy DataFrame labeled_90. "
        "Hãy chạy Cell 1 và Cell 2 trước."
    )

snapshots = (
    labeled_90
    .withColumn("snapshot_at", F.to_timestamp("snapshot_at"))
    .withColumn("snapshot_date", F.to_date("snapshot_date"))
    .withColumn("previous_purchase_at", F.to_timestamp("previous_purchase_at"))
    .withColumn("next_purchase_at", F.to_timestamp("next_purchase_at"))
    .withColumn("followup_end_90", F.to_timestamp("followup_end_90"))
    .withColumn("purchase_sequence", F.col("purchase_sequence").cast("int"))
    .withColumn(
        "days_since_previous_purchase",
        F.col("days_since_previous_purchase").cast("double")
    )
    .withColumn(
        "customer_tenure_days",
        F.col("customer_tenure_days").cast("double")
    )
    .withColumn(
        "avg_interpurchase_days_to_snapshot",
        F.col("avg_interpurchase_days_to_snapshot").cast("double")
    )
    .withColumn(
        "max_interpurchase_days_to_snapshot",
        F.col("max_interpurchase_days_to_snapshot").cast("double")
    )
    .withColumn("churn_90", F.col("churn_90").cast("int"))
    .select(
        "customer_unique_id",
        "snapshot_order_id",
        "purchase_sequence",
        "snapshot_at",
        "snapshot_date",
        "previous_purchase_at",
        "days_since_previous_purchase",
        "customer_tenure_days",
        "avg_interpurchase_days_to_snapshot",
        "max_interpurchase_days_to_snapshot",
        "followup_end_90",
        "churn_90"
    )
    .cache()
)

print("=" * 65)
print("SNAPSHOT DÙNG CHO FEATURE ENGINEERING")
print("=" * 65)
print(f"Số event có nhãn 90 ngày : {snapshots.count():,}")
print(
    f"Số khách                 : "
    f"{snapshots.select('customer_unique_id').distinct().count():,}"
)

print("\nPHÂN BỐ CHURN 90")
snapshots.groupBy("churn_90").agg(
    F.count("*").alias("so_event"),
    F.countDistinct("customer_unique_id").alias("so_khach")
).orderBy("churn_90").show(truncate=False)

# ============================================================
# 2. ĐỌC DỮ LIỆU NGUỒN
# ============================================================

orders = read_csv(orders_path)
customers = read_csv(customers_path).select(
    "customer_id", "customer_unique_id"
)
items = read_csv(items_path)
payments = read_csv(payments_path)
reviews = read_csv(reviews_path, multiline=True)

products = (
    read_csv(products_prepared_path)
    if products_prepared_path.exists()
    else None
)

payment_audit = (
    read_csv(payment_audit_path)
    if payment_audit_path.exists()
    else None
)

# ============================================================
# 3. ITEM / PRODUCT FACTS THEO ORDER
# ============================================================

item_source = items

if products is not None:
    item_source = item_source.join(
        products.select(
            "product_id",
            "product_category_name",
            "product_weight_g"
        ),
        "product_id",
        "left"
    )
else:
    item_source = (
        item_source
        .withColumn(
            "product_category_name",
            F.lit(None).cast("string")
        )
        .withColumn(
            "product_weight_g",
            F.lit(None).cast("double")
        )
    )

item_facts = (
    item_source
    .withColumn("price_num", F.col("price").cast("double"))
    .withColumn("freight_num", F.col("freight_value").cast("double"))
    .withColumn("weight_num", F.col("product_weight_g").cast("double"))
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.countDistinct("product_id").alias("distinct_products_in_order"),
        F.countDistinct("seller_id").alias("distinct_sellers_in_order"),
        F.countDistinct("product_category_name").alias("categories_in_order"),
        F.sum("price_num").alias("order_item_value"),
        F.sum("freight_num").alias("order_freight"),
        F.avg(
            F.when(F.col("weight_num") > 0, F.col("weight_num"))
        ).alias("mean_product_weight_g")
    )
)

# ============================================================
# 4. PAYMENT FACTS THEO ORDER
# ============================================================

payment_facts = (
    payments
    .withColumn("payment_num", F.col("payment_value").cast("double"))
    .withColumn(
        "installments_num",
        F.col("payment_installments").cast("int")
    )
    .groupBy("order_id")
    .agg(
        F.sum("payment_num").alias("order_payment"),
        F.count("*").alias("payment_lines"),
        F.countDistinct("payment_type").alias("payment_types_in_order"),
        F.max("installments_num").alias("max_installments_in_order"),
        F.max(
            F.when(F.col("payment_type") == "credit_card", 1).otherwise(0)
        ).alias("used_credit_card"),
        F.max(
            F.when(
                F.col("installments_num").isNull()
                | (F.col("installments_num") <= 0),
                1
            ).otherwise(0)
        ).alias("invalid_installments")
    )
)

# ============================================================
# 5. ORDER FACTS
# ============================================================

order_facts = (
    orders
    .filter(F.col("order_status") == "delivered")
    .join(customers, "customer_id", "inner")
    .select(
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
    .join(item_facts, "order_id", "left")
    .join(payment_facts, "order_id", "left")
    .withColumn(
        "purchase_at",
        F.to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "delivered_at",
        F.to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "estimated_delivery_at",
        F.to_timestamp("order_estimated_delivery_date")
    )
)

if payment_audit is not None:
    order_facts = order_facts.join(
        payment_audit.select(
            "order_id",
            "reconciliation_status"
        ),
        "order_id",
        "left"
    )
else:
    order_facts = order_facts.withColumn(
        "reconciliation_status",
        F.lit(None).cast("string")
    )

order_facts = (
    order_facts
    .withColumn(
        "delivery_days",
        F.when(
            F.col("delivered_at").isNotNull()
            & (F.col("delivered_at") >= F.col("purchase_at")),
            (
                F.unix_timestamp("delivered_at")
                - F.unix_timestamp("purchase_at")
            ) / 86400.0
        )
    )
    .withColumn(
        "late_flag",
        F.when(
            F.col("delivered_at").isNotNull()
            & F.col("estimated_delivery_at").isNotNull(),
            F.when(
                F.col("delivered_at") > F.col("estimated_delivery_at"),
                1
            ).otherwise(0)
        )
    )
    .withColumn(
        "payment_for_model",
        F.when(
            F.col("reconciliation_status").isNull()
            | (F.col("reconciliation_status") == "KHOP"),
            F.col("order_payment")
        )
    )
    .cache()
)

# ============================================================
# 6. HISTORY TẠI SNAPSHOT
# Chỉ sử dụng đơn hiện tại và các đơn đã xảy ra trước snapshot.
# ============================================================

history = (
    snapshots.alias("s")
    .join(
        order_facts.alias("o"),
        (
            F.col("s.customer_unique_id")
            == F.col("o.customer_unique_id")
        )
        & (
            (F.col("o.purchase_at") < F.col("s.snapshot_at"))
            | (
                (F.col("o.purchase_at") == F.col("s.snapshot_at"))
                & (
                    F.col("o.order_id")
                    <= F.col("s.snapshot_order_id")
                )
            )
        ),
        "inner"
    )
    .select(
        F.col("s.customer_unique_id").alias("customer_unique_id"),
        F.col("s.snapshot_order_id").alias("snapshot_order_id"),
        F.col("s.snapshot_at").alias("snapshot_at"),
        F.col("s.snapshot_date").alias("snapshot_date"),
        F.col("o.order_id").alias("history_order_id"),
        F.col("o.purchase_at").alias("purchase_at"),
        F.col("o.delivered_at").alias("delivered_at"),
        F.col("o.delivery_days").alias("delivery_days"),
        F.col("o.late_flag").alias("late_flag"),
        F.col("o.payment_for_model").alias("payment_for_model"),
        F.col("o.order_freight").alias("order_freight"),
        F.col("o.item_count").alias("item_count"),
        F.col("o.distinct_products_in_order")
        .alias("distinct_products_in_order"),
        F.col("o.distinct_sellers_in_order")
        .alias("distinct_sellers_in_order"),
        F.col("o.categories_in_order").alias("categories_in_order"),
        F.col("o.mean_product_weight_g")
        .alias("mean_product_weight_g"),
        F.col("o.payment_lines").alias("payment_lines"),
        F.col("o.max_installments_in_order")
        .alias("max_installments_in_order"),
        F.col("o.used_credit_card").alias("used_credit_card"),
        F.col("o.invalid_installments").alias("invalid_installments"),
        F.col("o.reconciliation_status")
        .alias("reconciliation_status")
    )
    .withColumn(
        "delivery_known",
        F.col("delivered_at").isNotNull()
        & (F.col("delivered_at") <= F.col("snapshot_at"))
    )
    .cache()
)

transaction_features = (
    history
    .groupBy(
        "customer_unique_id",
        "snapshot_order_id",
        "snapshot_at",
        "snapshot_date"
    )
    .agg(
        F.countDistinct("history_order_id").alias("frequency_orders"),

        F.sum(
            F.when(
                F.col("purchase_at")
                >= F.expr("snapshot_at - INTERVAL 30 DAYS"),
                1
            ).otherwise(0)
        ).alias("orders_last_30d"),

        F.sum(
            F.when(
                F.col("purchase_at")
                >= F.expr("snapshot_at - INTERVAL 90 DAYS"),
                1
            ).otherwise(0)
        ).alias("orders_last_90d"),

        F.sum(
            F.when(
                F.col("purchase_at")
                >= F.expr("snapshot_at - INTERVAL 180 DAYS"),
                1
            ).otherwise(0)
        ).alias("orders_last_180d"),

        F.sum("payment_for_model").alias("monetary_total"),
        F.avg("payment_for_model").alias("mean_order_value"),
        F.sum("order_freight").alias("freight_total"),

        F.sum(
            F.coalesce(F.col("item_count"), F.lit(0))
        ).alias("total_items"),

        F.sum(
            F.coalesce(
                F.col("distinct_products_in_order"),
                F.lit(0)
            )
        ).alias("sum_distinct_products_per_order"),

        F.sum(
            F.coalesce(
                F.col("distinct_sellers_in_order"),
                F.lit(0)
            )
        ).alias("sum_distinct_sellers_per_order"),

        F.sum(
            F.coalesce(
                F.col("categories_in_order"),
                F.lit(0)
            )
        ).alias("sum_categories_per_order"),

        F.avg("mean_product_weight_g")
        .alias("mean_order_product_weight_g"),

        F.count(
            F.when(F.col("delivery_known"), F.lit(1))
        ).alias("delivered_orders_known"),

        F.avg(
            F.when(
                F.col("delivery_known"),
                F.col("delivery_days")
            )
        ).alias("mean_delivery_days_known"),

        F.sum(
            F.when(
                F.col("delivery_known")
                & (F.col("late_flag") == 1),
                1
            ).otherwise(0)
        ).alias("late_orders_known"),

        F.sum(
            F.coalesce(F.col("payment_lines"), F.lit(0))
        ).alias("total_payment_lines"),

        F.max("max_installments_in_order")
        .alias("max_installments"),

        F.sum(
            F.coalesce(F.col("used_credit_card"), F.lit(0))
        ).alias("credit_card_orders"),

        F.sum(
            F.coalesce(F.col("invalid_installments"), F.lit(0))
        ).alias("orders_invalid_installments"),

        F.sum(
            F.when(
                F.col("reconciliation_status").isNotNull()
                & (F.col("reconciliation_status") != "KHOP"),
                1
            ).otherwise(0)
        ).alias("orders_payment_not_matched")
    )
)

# ============================================================
# 7. FEATURE CỦA CHÍNH ĐƠN SNAPSHOT
# ============================================================

current_order_features = (
    snapshots.alias("s")
    .join(
        order_facts.alias("o"),
        F.col("s.snapshot_order_id") == F.col("o.order_id"),
        "left"
    )
    .select(
        F.col("s.customer_unique_id").alias("customer_unique_id"),
        F.col("s.snapshot_order_id").alias("snapshot_order_id"),
        F.col("o.item_count").alias("current_item_count"),
        F.col("o.distinct_products_in_order")
        .alias("current_distinct_products"),
        F.col("o.distinct_sellers_in_order")
        .alias("current_distinct_sellers"),
        F.col("o.categories_in_order").alias("current_categories"),
        F.col("o.payment_for_model").alias("current_order_payment"),
        F.col("o.order_freight").alias("current_order_freight"),
        F.col("o.max_installments_in_order")
        .alias("current_max_installments"),
        F.col("o.used_credit_card")
        .alias("current_used_credit_card")
    )
)

# ============================================================
# 8. REVIEW CHỈ DÙNG NẾU ANSWER_AT <= SNAPSHOT
# ============================================================

unique_reviews = (
    reviews
    .filter(F.col("review_id").isNotNull())
    .withColumn(
        "answer_at",
        F.to_timestamp("review_answer_timestamp")
    )
    .withColumn(
        "score_num",
        F.col("review_score").cast("int")
    )
    .groupBy("review_id")
    .agg(
        F.countDistinct("order_id").alias("linked_orders"),
        F.min("order_id").alias("order_id"),
        F.countDistinct("score_num").alias("distinct_scores"),
        F.min("score_num").alias("review_score"),
        F.min("answer_at").alias("answer_at_min"),
        F.max("answer_at").alias("answer_at_max")
    )
    .filter(
        (F.col("linked_orders") == 1)
        & (F.col("distinct_scores") == 1)
        & F.col("review_score").between(1, 5)
        & F.col("answer_at_min").isNotNull()
        & (F.col("answer_at_min") == F.col("answer_at_max"))
    )
    .select(
        "review_id",
        "order_id",
        "review_score",
        F.col("answer_at_min").alias("answer_at")
    )
)

review_features = (
    history
    .select(
        "customer_unique_id",
        "snapshot_order_id",
        "snapshot_at",
        F.col("history_order_id").alias("order_id")
    )
    .join(
        unique_reviews,
        "order_id",
        "inner"
    )
    .filter(
        F.col("answer_at") <= F.col("snapshot_at")
    )
    .groupBy(
        "customer_unique_id",
        "snapshot_order_id"
    )
    .agg(
        F.countDistinct("review_id")
        .alias("known_review_count"),

        F.avg("review_score")
        .alias("mean_known_review_score"),

        F.sum(
            F.when(
                F.col("review_score") <= 2,
                1
            ).otherwise(0)
        ).alias("low_score_review_count")
    )
)

# ============================================================
# 9. GỘP FEATURE + TEMPORAL SPLIT
# ============================================================

features = (
    snapshots
    .join(
        transaction_features,
        [
            "customer_unique_id",
            "snapshot_order_id",
            "snapshot_at",
            "snapshot_date"
        ],
        "left"
    )
    .join(
        current_order_features,
        ["customer_unique_id", "snapshot_order_id"],
        "left"
    )
    .join(
        review_features,
        ["customer_unique_id", "snapshot_order_id"],
        "left"
    )
    .withColumn(
        "prior_orders_count",
        F.col("purchase_sequence") - 1
    )
    .withColumn(
        "known_review_count",
        F.coalesce(
            F.col("known_review_count"),
            F.lit(0)
        )
    )
    .withColumn(
        "low_score_review_count",
        F.coalesce(
            F.col("low_score_review_count"),
            F.lit(0)
        )
    )
    .withColumn(
        "temporal_split",
        F.when(
            F.col("snapshot_date")
            <= F.to_date(F.lit(TRAIN_END)),
            "train"
        )
        .when(
            F.col("snapshot_date")
            >= F.to_date(F.lit(TEST_START)),
            "test"
        )
        .otherwise("embargo")
    )
    .cache()
)

# ============================================================
# 10. KIỂM TRA SPLIT VÀ LEAKAGE
# ============================================================

train_label_end = (
    features
    .filter(F.col("temporal_split") == "train")
    .agg(
        F.max("followup_end_90")
        .alias("train_label_end")
    )
    .first()["train_label_end"]
)

test_start_at = (
    features
    .filter(F.col("temporal_split") == "test")
    .agg(
        F.min("snapshot_at")
        .alias("test_start_at")
    )
    .first()["test_start_at"]
)

if train_label_end is None or test_start_at is None:
    raise ValueError(
        "Train hoặc test rỗng sau temporal split."
    )

if train_label_end >= test_start_at:
    raise ValueError(
        f"Leakage thời gian: train label kết thúc "
        f"{train_label_end}, test bắt đầu {test_start_at}"
    )

frequency_mismatch = (
    features
    .filter(
        F.col("frequency_orders")
        != F.col("purchase_sequence")
    )
    .count()
)

duplicate_feature_keys = (
    features
    .groupBy(
        "customer_unique_id",
        "snapshot_order_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nKIỂM TRA FEATURE")
print(f"Feature rows                         : {features.count():,}")
print(f"Duplicate customer + snapshot_order : {duplicate_feature_keys:,}")
print(f"frequency_orders != purchase_sequence: {frequency_mismatch:,}")
print(f"Train label end                     : {train_label_end}")
print(f"Test start                          : {test_start_at}")

if duplicate_feature_keys > 0 or frequency_mismatch > 0:
    raise ValueError("Kiểm tra feature chưa đạt.")

# ============================================================
# 11. LƯU FEATURE DATASET
# ============================================================

(
    features
    .repartition(8, "snapshot_date")
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(feature_output_dir))
)

print("\nĐã lưu feature dataset:")
print(feature_output_dir)

print("\nPHÂN BỐ TRAIN / EMBARGO / TEST")
(
    features
    .groupBy("temporal_split")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct(
            "customer_unique_id"
        ).alias("so_khach"),
        F.sum(
            F.when(
                F.col("churn_90") == 1,
                1
            ).otherwise(0)
        ).alias("so_churn"),
        F.sum(
            F.when(
                F.col("churn_90") == 0,
                1
            ).otherwise(0)
        ).alias("so_non_churn"),
        F.round(
            100 * F.avg("churn_90"),
            2
        ).alias("ty_le_churn_pct")
    )
    .orderBy("temporal_split")
    .show(truncate=False)
)

history.unpersist()
order_facts.unpersist()
snapshots.unpersist()
features.unpersist()

SNAPSHOT DÙNG CHO FEATURE ENGINEERING
Số event có nhãn 90 ngày : 78,019
Số khách                 : 75,513

PHÂN BỐ CHURN 90
+--------+--------+--------+
|churn_90|so_event|so_khach|
+--------+--------+--------+
|0       |2148    |1953    |
|1       |75871   |75227   |
+--------+--------+--------+


KIỂM TRA FEATURE
Feature rows                         : 78,019
Duplicate customer + snapshot_order : 0
frequency_orders != purchase_sequence: 0
Train label end                     : 2018-01-30 23:43:51
Test start                          : 2018-02-01 00:24:55

Đã lưu feature dataset:
/content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_customer_churn_features_temporal

PHÂN BỐ TRAIN / EMBARGO / TEST
+--------------+--------+--------+--------+------------+---------------+
|temporal_split|so_event|so_khach|so_churn|so_non_churn|ty_le_churn_pct|
+--------------+--------+--------+--------+------------+---------------+
|embargo       |19763   |19416   |19313   |450         |97.72  

DataFrame[customer_unique_id: string, snapshot_order_id: string, snapshot_at: timestamp, snapshot_date: date, purchase_sequence: int, previous_purchase_at: timestamp, days_since_previous_purchase: double, customer_tenure_days: double, avg_interpurchase_days_to_snapshot: double, max_interpurchase_days_to_snapshot: double, followup_end_90: timestamp, churn_90: int, frequency_orders: bigint, orders_last_30d: bigint, orders_last_90d: bigint, orders_last_180d: bigint, monetary_total: double, mean_order_value: double, freight_total: double, total_items: bigint, sum_distinct_products_per_order: bigint, sum_distinct_sellers_per_order: bigint, sum_categories_per_order: bigint, mean_order_product_weight_g: double, delivered_orders_known: bigint, mean_delivery_days_known: double, late_orders_known: bigint, total_payment_lines: bigint, max_installments: int, credit_card_orders: bigint, orders_invalid_installments: bigint, orders_payment_not_matched: bigint, current_item_count: bigint, current_dist

### Cell 4 — Kiểm tra cuối trước khi Model Training

- Kiểm tra khóa observation và target `churn_90`.
- Kiểm tra class distribution ở purchase-event level.
- Kiểm tra số event/khách và overlap khách giữa train/test.
- Kiểm tra missing, zero, quantile và feature hằng số.


In [8]:
from pyspark.sql import functions as F

feature_dir = processed_dir / "olist_customer_churn_features_temporal"

feature_columns = [
    "purchase_sequence",
    "prior_orders_count",
    "days_since_previous_purchase",
    "customer_tenure_days",
    "avg_interpurchase_days_to_snapshot",
    "max_interpurchase_days_to_snapshot",
    "frequency_orders",
    "orders_last_30d",
    "orders_last_90d",
    "orders_last_180d",
    "monetary_total",
    "mean_order_value",
    "freight_total",
    "total_items",
    "sum_distinct_products_per_order",
    "sum_distinct_sellers_per_order",
    "sum_categories_per_order",
    "mean_order_product_weight_g",
    "delivered_orders_known",
    "mean_delivery_days_known",
    "late_orders_known",
    "total_payment_lines",
    "max_installments",
    "credit_card_orders",
    "orders_invalid_installments",
    "orders_payment_not_matched",
    "known_review_count",
    "mean_known_review_score",
    "low_score_review_count",
    "current_item_count",
    "current_distinct_products",
    "current_distinct_sellers",
    "current_categories",
    "current_order_payment",
    "current_order_freight",
    "current_max_installments",
    "current_used_credit_card"
]

raw = spark.read.option("header", True).csv(str(feature_dir))

required = [
    "customer_unique_id",
    "snapshot_order_id",
    "snapshot_at",
    "snapshot_date",
    "followup_end_90",
    "temporal_split",
    "churn_90",
] + feature_columns

missing_columns = sorted(set(required) - set(raw.columns))
if missing_columns:
    raise ValueError(f"Thiếu cột: {missing_columns}")

data = (
    raw
    .select(
        "customer_unique_id",
        "snapshot_order_id",
        F.to_timestamp("snapshot_at").alias("snapshot_at"),
        F.to_date("snapshot_date").alias("snapshot_date"),
        F.to_timestamp("followup_end_90").alias("followup_end_90"),
        "temporal_split",
        F.col("churn_90").cast("int").alias("churn_90"),
        *[
            F.expr(f"try_cast(`{name}` as double)").alias(name)
            for name in feature_columns
        ]
    )
    .filter(F.col("temporal_split").isin("train", "test"))
    .cache()
)

# ============================================================
# 1. KEY + LABEL CHECK
# ============================================================

duplicate_keys = (
    data
    .groupBy("customer_unique_id", "snapshot_order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_labels = data.filter(~F.col("churn_90").isin(0, 1)).count()

print("Duplicate customer + snapshot_order_id:", duplicate_keys)
print("Nhãn churn_90 ngoài {0,1}             :", invalid_labels)

if duplicate_keys > 0 or invalid_labels > 0:
    raise ValueError("Key hoặc target chưa đạt")

# ============================================================
# 2. CLASS DISTRIBUTION
# ============================================================

class_summary = (
    data
    .groupBy("temporal_split")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct("customer_unique_id").alias("so_khach"),
        F.sum(F.when(F.col("churn_90") == 1, 1).otherwise(0)).alias("so_churn"),
        F.sum(F.when(F.col("churn_90") == 0, 1).otherwise(0)).alias("so_non_churn")
    )
    .withColumn("ty_le_churn_pct", F.round(100 * F.col("so_churn") / F.col("so_event"), 2))
    .withColumn("ty_le_non_churn_pct", F.round(100 * F.col("so_non_churn") / F.col("so_event"), 2))
    .withColumn(
        "majority_baseline_accuracy_pct",
        F.round(
            100 * F.greatest(F.col("so_churn"), F.col("so_non_churn")) / F.col("so_event"),
            2
        )
    )
    .orderBy("temporal_split")
)

print("PHÂN BỐ NHÃN CHURN_180 — PURCHASE EVENT LEVEL")
class_summary.show(truncate=False)

# ============================================================
# 3. EVENT / CUSTOMER VÀ TRAIN–TEST OVERLAP
# ============================================================

event_per_customer = (
    data
    .groupBy("customer_unique_id")
    .agg(F.count("*").alias("so_event"))
)

print("THỐNG KÊ EVENT / KHÁCH")
(
    event_per_customer
    .agg(
        F.count("*").alias("so_khach"),
        F.round(F.avg("so_event"), 2).alias("event_trung_binh_moi_khach"),
        F.min("so_event").alias("event_min"),
        F.max("so_event").alias("event_max")
    )
    .show(truncate=False)
)

train_customers = (
    data.filter(F.col("temporal_split") == "train")
    .select("customer_unique_id").distinct()
)

test_customers = (
    data.filter(F.col("temporal_split") == "test")
    .select("customer_unique_id").distinct()
)

overlap_customers = train_customers.join(test_customers, "customer_unique_id", "inner").count()

print("Số khách train                     :", train_customers.count())
print("Số khách test                      :", test_customers.count())
print("Số khách xuất hiện cả train và test:", overlap_customers)

# ============================================================
# 4. MISSING / ZERO / QUANTILE / CONSTANT FEATURE
# ============================================================

aggregations = [F.count("*").alias("n")]

for name in feature_columns:
    c = F.col(name)
    missing = c.isNull() | F.isnan(c)
    aggregations.extend([
        F.sum(F.when(missing, 1).otherwise(0)).alias(f"{name}__missing"),
        F.sum(F.when((~missing) & (c == 0), 1).otherwise(0)).alias(f"{name}__zero"),
        F.min(F.when(~missing, c)).alias(f"{name}__min"),
        F.max(F.when(~missing, c)).alias(f"{name}__max"),
        F.percentile_approx(
            F.when(~missing, c),
            [0.25, 0.50, 0.75, 0.90],
            1000
        ).alias(f"{name}__quantiles")
    ])

stats = {
    row["temporal_split"]: row.asDict()
    for row in data.groupBy("temporal_split").agg(*aggregations).collect()
}

if set(stats) != {"train", "test"}:
    raise ValueError(f"Cần cả train và test, hiện có: {sorted(stats)}")

rows = []
constant_features = []

for name in feature_columns:
    tr = stats["train"]
    te = stats["test"]

    tr_n, te_n = tr["n"], te["n"]
    tr_miss = 100 * tr[f"{name}__missing"] / tr_n
    te_miss = 100 * te[f"{name}__missing"] / te_n

    tr_valid = tr_n - tr[f"{name}__missing"]
    te_valid = te_n - te[f"{name}__missing"]

    tr_zero = 100 * tr[f"{name}__zero"] / tr_valid if tr_valid else None
    te_zero = 100 * te[f"{name}__zero"] / te_valid if te_valid else None

    tr_q = tr[f"{name}__quantiles"] or [None, None, None, None]
    te_q = te[f"{name}__quantiles"] or [None, None, None, None]

    if tr[f"{name}__min"] == tr[f"{name}__max"] and tr_valid > 0:
        constant_features.append(name)

    rows.append((
        name,
        round(tr_miss, 2),
        round(te_miss, 2),
        round(te_miss - tr_miss, 2),
        round(tr_zero, 2) if tr_zero is not None else None,
        round(te_zero, 2) if te_zero is not None else None,
        *[round(x, 2) if x is not None else None for x in tr_q],
        *[round(x, 2) if x is not None else None for x in te_q],
    ))

comparison = spark.createDataFrame(
    rows,
    """
    feature string,
    train_missing_pct double,
    test_missing_pct double,
    missing_gap_pp double,
    train_zero_pct double,
    test_zero_pct double,
    train_p25 double,
    train_p50 double,
    train_p75 double,
    train_p90 double,
    test_p25 double,
    test_p50 double,
    test_p75 double,
    test_p90 double
    """
)

print("MISSING / ZERO / QUANTILE: TRAIN SO VỚI TEST")
comparison.orderBy(F.desc(F.abs(F.col("missing_gap_pp"))), "feature").show(
    len(feature_columns), truncate=False
)

print("Feature hằng số trong train (nên loại trước model):", constant_features)

print("=" * 72)
print("KIỂM TRA CUỐI")
print("=" * 72)
print("Duplicate key        :", duplicate_keys)
print("Invalid target       :", invalid_labels)
print("Train/test overlap   :", overlap_customers)
print("Constant train feats :", constant_features)
print("Đơn vị dữ liệu       : 1 dòng = 1 lần mua thực tế của 1 khách")

data.unpersist()


Duplicate customer + snapshot_order_id: 0
Nhãn churn_90 ngoài {0,1}             : 0
PHÂN BỐ NHÃN CHURN_180 — PURCHASE EVENT LEVEL
+--------------+--------+--------+--------+------------+---------------+-------------------+------------------------------+
|temporal_split|so_event|so_khach|so_churn|so_non_churn|ty_le_churn_pct|ty_le_non_churn_pct|majority_baseline_accuracy_pct|
+--------------+--------+--------+--------+------------+---------------+-------------------+------------------------------+
|test          |27255   |26714   |26423   |832         |96.95          |3.05               |96.95                         |
|train         |31001   |30096   |30135   |866         |97.21          |2.79               |97.21                         |
+--------------+--------+--------+--------+------------+---------------+-------------------+------------------------------+

THỐNG KÊ EVENT / KHÁCH
+--------+--------------------------+---------+---------+
|so_khach|event_trung_binh_moi_khach|event_m

DataFrame[customer_unique_id: string, snapshot_order_id: string, snapshot_at: timestamp, snapshot_date: date, followup_end_90: timestamp, temporal_split: string, churn_90: int, purchase_sequence: double, prior_orders_count: double, days_since_previous_purchase: double, customer_tenure_days: double, avg_interpurchase_days_to_snapshot: double, max_interpurchase_days_to_snapshot: double, frequency_orders: double, orders_last_30d: double, orders_last_90d: double, orders_last_180d: double, monetary_total: double, mean_order_value: double, freight_total: double, total_items: double, sum_distinct_products_per_order: double, sum_distinct_sellers_per_order: double, sum_categories_per_order: double, mean_order_product_weight_g: double, delivered_orders_known: double, mean_delivery_days_known: double, late_orders_known: double, total_payment_lines: double, max_installments: double, credit_card_orders: double, orders_invalid_installments: double, orders_payment_not_matched: double, known_review_co